# L19 — Resource Schedules and Shift Patterns

**Module**: M06 | **Chapter**: 8 | **Lecture**: L19

## Learning Objectives
By the end of this notebook you will be able to:
1. Implement shift-based server availability using SimPy capacity changes.
2. Model shift-change handoffs and stagger patterns.
3. Analyse the effect of shift boundaries on queue build-up.
4. Implement scheduled maintenance windows that remove servers from service.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**
---

In [ ]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

## 1. Shift Schedule as a Resource Capacity Timeline

In SimPy, `Resource.capacity` can be changed at runtime.  
This allows modelling shift patterns: more servers during peak hours, fewer overnight.

In [ ]:
# Define a shift schedule: (start_hour, end_hour, capacity)
# Operating hours 6:00–22:00; shift change at noon
SHIFT_SCHEDULE = [
    (0,  6,  0),   # closed overnight
    (6,  10, 1),   # early shift: 1 server
    (10, 14, 3),   # mid-morning: 3 servers (peak)
    (14, 18, 3),   # afternoon: 3 servers (peak)
    (18, 22, 2),   # evening: 2 servers
    (22, 24, 0),   # closed
]

def capacity_at(t_hr: float, schedule=SHIFT_SCHEDULE) -> int:
    """Return server capacity at time t_hr (hour of day, fractional)."""
    h = t_hr % 24
    for start, end, cap in schedule:
        if start <= h < end:
            return cap
    return 0

# Plot the schedule
hours = np.linspace(0, 48, 1000)   # 2 days
caps  = [capacity_at(h) for h in hours]

fig, ax = plt.subplots(figsize=(12, 3))
ax.step(hours, caps, where='post', color='steelblue', lw=2)
ax.fill_between(hours, caps, step='post', alpha=0.3, color='steelblue')
ax.set_xlabel('Hour')
ax.set_ylabel('Active servers')
ax.set_title('Shift schedule (2-day view)')
ax.set_xticks(range(0, 50, 4))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def shift_schedule_sim(lam_fn, mu: float, schedule,
                       sim_days: int = 30, seed: int = 0) -> pd.DataFrame:
    """
    Simulate a queue with a scheduled resource capacity profile.

    lam_fn   : callable(t_hr) -> arrival rate at that hour
    mu       : service rate per server
    schedule : list of (start_hr, end_hr, capacity) tuples
    """
    rng = np.random.default_rng(seed)
    env = simpy.Environment()

    # Start with capacity = schedule[0 hour]
    initial_cap = capacity_at(0, schedule)
    server = simpy.Resource(env, capacity=max(initial_cap, 1))
    server._capacity = initial_cap  # allow 0 for closed periods

    records = []
    lam_star = max(lam_fn(h) for h in range(24))

    def capacity_manager():
        """Process that updates server capacity at shift boundaries."""
        day = 0
        while day < sim_days:
            for start_hr, end_hr, cap in schedule:
                t_start = day * 24 + start_hr
                t_end   = day * 24 + end_hr
                if env.now < t_start:
                    yield env.timeout(t_start - env.now)
                # Change capacity
                server._capacity = cap
                # Trigger pending requests if capacity increased
                server._trigger_put(None)
            day += 1

    def patient():
        arrival  = env.now
        hour_day = arrival % 24
        # Don't arrive when closed
        if capacity_at(hour_day, schedule) == 0:
            return
        with server.request() as req:
            yield req
            wait = env.now - arrival
            svc  = rng.exponential(1.0 / mu)
            yield env.timeout(svc)
        records.append({'hour': hour_day, 'wait_min': wait * 60,
                         'cap': capacity_at(hour_day, schedule)})

    def arrivals():
        while True:
            yield env.timeout(rng.exponential(1.0 / lam_star))
            lam_now = lam_fn(env.now % 24)
            if rng.random() < lam_now / lam_star:
                env.process(patient())

    env.process(capacity_manager())
    env.process(arrivals())
    env.run(until=sim_days * 24)

    return pd.DataFrame(records)


# Arrival rate profile (patients/hr by hour of day)
rate_by_hour = [0]*6 + [2, 4, 8, 10, 9, 8, 7, 8, 9, 10, 8, 6, 4, 3, 0, 0, 0, 0]
def lam_fn(h):
    return rate_by_hour[int(h) % 24]

df_sched = shift_schedule_sim(lam_fn, mu=4.0, schedule=SHIFT_SCHEDULE, sim_days=60, seed=42)
print(f"Records: {len(df_sched):,}")
print(f"Overall mean wait: {df_sched['wait_min'].mean():.2f} min")
print()
print("Mean wait by shift:")
print(df_sched.groupby('cap')['wait_min'].agg(['mean','count']).round(2))

## 2. Shift-Change Queue Build-Up

At shift boundaries, queues can spike if there is a hand-off delay or a capacity drop.

In [ ]:
# Hourly wait breakdown
hourly_wait = df_sched.groupby(df_sched['hour'].astype(int))['wait_min'].mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.bar(range(24), [lam_fn(h) for h in range(24)],
        color='steelblue', alpha=0.6, label='Arrival rate')
ax1_right = ax1.twinx()
ax1_right.step(range(25), [capacity_at(h) for h in range(25)],
               color='orange', lw=2, where='post', label='Server capacity')
ax1.set_ylabel('Arrival rate (patients/hr)')
ax1_right.set_ylabel('Active servers')
ax1.set_title('Arrival rate and server capacity')

ax2.bar(hourly_wait.index, hourly_wait.values, color='tomato', alpha=0.7)
ax2.set_xlabel('Hour of day')
ax2.set_ylabel('Mean wait (min)')
ax2.set_title('Mean patient wait by hour')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Staggered Shifts: Avoiding the Hand-Off Spike

Instead of all servers changing at the same time, staggering shifts so the incoming shift arrives 30 min early reduces the queue spike at shift change.

In [ ]:
# Staggered schedule: 30-min overlap at each shift change
STAGGERED = [
    (0,    6,    0),
    (6,    10,   1),
    (9.5,  13.5, 4),   # incoming shift overlaps for 0.5 hr
    (10,   14,   3),   # outgoing shift ends at 10
    (13.5, 17.5, 4),   # overlap at afternoon shift
    (14,   18,   3),
    (17.5, 21.5, 3),   # overlap evening
    (18,   22,   2),
    (22,   24,   0),
]

# For simplicity: just demonstrate the idea conceptually
print("Staggered shift principle:")
print("  At shift change time T:")
print("    - Incoming staff arrive at T-0.5 hr (overlap period)")
print("    - Outgoing staff leave at T")
print("    - For 0.5 hr, capacity = incoming + outgoing (temporarily higher)")
print("    - This absorbs queued patients before handoff")
print()
print("Result: smoother wait times, at cost of ~0.5 hr extra labour per shift change.")
print("Simulate by adding capacity=c_out+c_in during overlap window.")

## 4. Scheduled Maintenance Windows

Some resources have mandatory downtime (equipment maintenance, IT updates).  
This is a capacity drop to 0 on a fixed schedule — distinct from random breakdowns.

In [ ]:
def machine_with_pm(lam, mu, pm_interval, pm_duration,
                    sim_time=10_000, seed=0):
    """
    Single machine with periodic scheduled maintenance (PM).
    PM occurs every pm_interval time units and lasts pm_duration.
    During PM, no new jobs start; in-progress jobs wait.
    """
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    machine = simpy.Resource(env, capacity=1)
    pm_active = [False]
    waits, sojourns = [], []

    def job():
        arrival = env.now
        with machine.request() as req:
            yield req
            # Wait out any active PM
            while pm_active[0]:
                yield env.timeout(0.01)
            wait = env.now - arrival
            svc  = rng.exponential(1.0 / mu)
            yield env.timeout(svc)
        waits.append(wait)
        sojourns.append(wait + svc)

    def pm_schedule():
        while True:
            yield env.timeout(pm_interval)
            pm_active[0] = True
            yield env.timeout(pm_duration)
            pm_active[0] = False

    def arrivals():
        while True:
            yield env.timeout(rng.exponential(1.0 / lam))
            env.process(job())

    env.process(arrivals())
    env.process(pm_schedule())
    env.run(until=sim_time)

    # Availability
    n_pm_windows = int(sim_time / pm_interval)
    avail = 1 - n_pm_windows * pm_duration / sim_time
    return {'Wq': np.mean(waits), 'W': np.mean(sojourns),
            'availability_approx': avail, 'n_served': len(waits)}


lam_m, mu_m = 3.0, 4.0
print(f"Base case (no PM): M/M/1 W = {1/(mu_m-lam_m):.4f}")
print()
print(f"{'PM interval':>12s}  {'PM duration':>12s}  {'Availability':>14s}  {'W':>10s}  {'Wq':>10s}")
print('-' * 65)
for pm_int, pm_dur in [(100, 2), (50, 2), (20, 2), (100, 10)]:
    r = machine_with_pm(lam_m, mu_m, pm_int, pm_dur, seed=0)
    print(f"{pm_int:>12d}  {pm_dur:>12d}  "
          f"{r['availability_approx']:>14.3f}  "
          f"{r['W']:>10.4f}  {r['Wq']:>10.4f}")

---
## Try It Yourself

1. **Optimal shift overlap**: Simulate a 2-shift system (day shift 8–16, evening shift 16–24) with λ=8/hr and μ=5/hr/server. Vary the overlap duration from 0 to 60 min. Plot mean queue length at shift change as a function of overlap. What overlap minimises the spike at cost of < 5% extra labour?

2. **Schedule optimisation**: Given a known hourly demand pattern and a fixed total of 16 server-hours per day, find the schedule allocation that minimises the maximum hourly mean wait. (Hint: use simulation + a simple grid search over schedule configurations.)

3. **Planned vs. unplanned downtime**: A machine has MTTF=100 hr (random failure) and scheduled PM every 50 hr for 1 hr. Compare total downtime and mean wait under three policies: (a) no PM, just repair failures; (b) PM every 50 hr; (c) PM every 25 hr. What trade-off emerges?